<a href="https://colab.research.google.com/github/silviamoya/03MIAR-Algoritmos-Optimizacion/blob/main/SEMINARIO/Trabajo_Practico_Grupo36.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Seminario<br>
**Nombre y Apellidos:** Silvia Moya Botella<br>
**Grupo:** 36. Paula Mejías Hernández y Silvia Moya Botella<br>
**Url:** https://github.com/silviamoya/03MIAR-Algoritmos-Optimizacion/blob/main/SEMINARIO/Trabajo_Practico_Grupo36.ipynb <br>
**Problema:**
> **1. Sesiones de doblaje**<br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

**Descripción del problema:**<br>

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día.<BR>
El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. Los datos son:
- Número de actores: 10
- Número de tomas : 30  
- Actores/Tomas : https://bit.ly/36D8IuK
>- 1 indica que el actor participa en la toma
>- 0 en caso contrario


---





                                        

## **Solución del problema**:
Distribuimos nuestra respuesta en tres etapas:

1. **Carga y preparación de los datos.**
2. **Backtracking con poda:**
- una primera versión sencilla;
- una segunda versión con cálculo incremental.
3. **Greedy con búsqueda local** como solución práctica.


### 1. Carga y Preparación de Datos

In [16]:
#Librerías necesarias para la lectura de datos y los cálculos matemáticos
import pandas as pd
import math
import time



In [17]:
sheet_id = "1Ipn6IrbQP4ax8zOnivdBIw2lN0JISkJG4fXndYd27U0"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

# Cargamos el CSV proporcionado utilizando la segunda fila como cabecera
df = pd.read_csv(url, header=1)

# Eliminamos las filas y columnas completamente vacías
df = df.dropna(axis=1, how="all")

# Mostramos las 5 primeras filas
df.head()

,Toma,1,2,3,4,5,6,7,8,9,10,Total
0,1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,5.0
1,2,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0
2,3,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,3.0
3,4,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,4.0
4,5,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,3.0


Se trata de un tipo de problema de planificación de tareas con restricciones y minimización de coste.

Restricciones:

*   Cada toma debe asignarse a un día
*   En cada día se pueden grabar 6 tomas como máximo
*   Si una toma usa un actor, entonces ese actor debe estar convocado ese día.

Para intentar solucionar este problema vamos a utlizar las siguientes técnicas:

**Backtracking con poda ->** Vamos asignando tomas a días, comprobamos restricciones ( máximo 6 tomas al día), si una asignación ya es peor que la mejor encontrada, la descarta.



### 2. Backtracking con poda

In [18]:
# En la matriz, cada fila representa una toma y cada columna un actor.
# Se eliminan las columnas "Total" y "Toma" y la fila "Total",
# ya que no forman parte de los datos de entrada del algoritmo.

tomas_actores = df.iloc[1:-1, 1:-1].values.tolist()

# Número máximo de tomas que pueden programarse en un mismo día.
Max_tomas_dias = 6

# Dimensiones del problema.
num_tomas = len(tomas_actores)
num_actores = len(tomas_actores[0])

print(f"Número de tomas: {num_tomas}")
print(f"Número de actores: {num_actores}")

Número de tomas: 30
Número de actores: 10


In [ ]:
# Inicialización de la mejor solución encontrada

mejor_coste = math.inf # Se inicializa con infinito para que cualquier solución válida tenga un coste menor.
mejor_plan = None # Aun no se ha encontrado ninguna planificación completa.

def coste_plan(dias):

    """
    Calcula el coste total de una planificación.

    El coste de cada día corresponde al número de actores distintos
    que participan en las tomas programadas durante esa jornada.
    """
    coste = 0

    for dia in dias:
        actores_del_dia = set()

    # Reunimos los actores que participan en alguna toma del día.
        for toma in dia:
            for actor, aparece in enumerate(tomas_actores[toma]):
                if aparece == 1:
                    actores_del_dia.add(actor)

    # Cada actor se contabiliza una sola vez por jornada.
        coste += len(actores_del_dia)

    return coste

def backtracking(toma_actual, dias):

    """
    Construye recursivamente una planificación de las tomas.
    """

    global mejor_coste, mejor_plan

    # Caso base: todas las tomas han sido asignadas.
    if toma_actual == num_tomas:
        coste = coste_plan(dias)

        # Actualizamos la mejor solución si la planificación actual es más barata.
        if coste < mejor_coste:
            mejor_coste = coste
            mejor_plan = [dia.copy() for dia in dias]

        return

    # Poda
    coste_actual = coste_plan(dias)
    if coste_actual >= mejor_coste:
        return

    for dia in dias:
        if len(dia) < Max_tomas_dias:
            dia.append(toma_actual)
            backtracking(toma_actual + 1, dias)
            dia.pop()

    dias.append([toma_actual])
    backtracking(toma_actual +1, dias)
    dias.pop()

backtracking(0,[])

print("Mejor coste:", mejor_coste)
print("Mejor planificacion:")

for i,dia in enumerate(mejor_plan, start=1):
    print(f"Día {i}: tomas {dia}")

KeyboardInterrupt: 

Usando la técnica de backtracking con poda se exploran todas las soluciones posibles. Aunque este método garantiza encontrar la solución óptima, su coste computacional es exponencial, por lo que el tiempo de ejecución aumenta considerablemente cuando el número de tomas es elevado. En nuestro caso, con las 30 tomas del problema, la ejecución no finaliza en un tiempo razonable.

Para mejorar la eficiencia, en lugar de recalcular el coste completo de la planificación en cada llamada recursiva, se almacenarán los actores asignados a cada día mediante conjuntos (sets). De este modo, al añadir una nueva toma solo será necesario actualizar la información del día afectado, reduciendo el número de operaciones realizadas y mejorando el rendimiento del algoritmo.

In [27]:
# Convertimos la matriz binaria en una lista de conjuntos.
# Cada conjunto contiene los actores que participan en una toma.
actores_por_toma = []

# Agregamos los actores por toma en la lista
for toma in tomas_actores:
    actores = set()

    for actor, aparece in enumerate(toma):
        if aparece == 1:
            actores.add(actor)

    actores_por_toma.append(actores)

# Número mínimo de días necesarios
num_dias = math.ceil(num_tomas / Max_tomas_dias)

# Empezamos con un valor de infinito como el mejor coste
mejor_coste = math.inf
mejor_plan = None

# Definimos la función de backtracking con poda
def backtracking(toma_actual, dias, actores_por_dia, coste_actual):
    global mejor_coste, mejor_plan

    # Cuando ya hayamos colocado todas las tomas comprobamos si es la mejor planificación
    if toma_actual == num_tomas:
        if coste_actual < mejor_coste:
            mejor_coste = coste_actual
            mejor_plan = [dia.copy() for dia in dias]
        return

    # Poda
    if coste_actual >= mejor_coste:
        return

    # Actores que participan en la toma actual
    actores_toma = actores_por_toma[toma_actual]

    # Intentar colocar la toma en un día existente
    for i in range(len(dias)):

        if len(dias[i]) < Max_tomas_dias:

            actores_antes = actores_por_dia[i].copy()

            actores_nuevos = actores_toma - actores_por_dia[i]
            incremento_coste = len(actores_nuevos)

            dias[i].append(toma_actual)
            actores_por_dia[i].update(actores_toma)

            backtracking(
                toma_actual + 1,
                dias,
                actores_por_dia,
                coste_actual + incremento_coste
            )

            # Deshacer cambios
            dias[i].pop()
            actores_por_dia[i] = actores_antes

    # Crear un nuevo día solo si todavía no se ha alcanzado el máximo
    if len(dias) < num_dias:

        dias.append([toma_actual])
        actores_por_dia.append(actores_toma.copy())

        backtracking(
            toma_actual + 1,
            dias,
            actores_por_dia,
            coste_actual + len(actores_toma)
        )

        # Deshacer día nuevo
        dias.pop()
        actores_por_dia.pop()

backtracking(0, [], [], 0)

print("Mejor coste:", mejor_coste)
print("Mejor planificación:")

for i, dia in enumerate(mejor_plan, start=1):
    print(f"Día {i}: tomas {[toma + 1 for toma in dia]}")

Mejor coste: 26
Mejor planificación:
Día 1: tomas [1, 4, 5, 6, 10, 12]
Día 2: tomas [2, 3, 7, 14, 20, 28]
Día 3: tomas [8, 15, 24, 27, 29, 30]
Día 4: tomas [9, 11, 19, 21, 25, 26]
Día 5: tomas [13, 16, 17, 18, 22, 23]


In [28]:
print(mejor_plan)
print(mejor_coste)
print(coste_plan(mejor_plan))

[[0, 3, 4, 5, 9, 11], [1, 2, 6, 13, 19, 27], [7, 14, 23, 26, 28, 29], [8, 10, 18, 20, 24, 25], [12, 15, 16, 17, 21, 22]]
26
26


Para comparar, vamos a utilizar una técnica que no garantice encontrar el óptimo pero que el coste operacional sea más bajo.

Utilizaremos una mezcla de dos técnicas. Iniciaremos con Greedy que irá colocando cada toma en el día donde añada menos actores nuevos y si no cabe en ningún día, creará un día nuevo.
Luego usaremos búsqueda local para intentar mejorar la solución moviendo tomas entre días, incluso intercambiar dos tomas entre días. Si el coste baja, aceptará el cambio.


### 3. Greedy con búsqueda local como solución práctica.


In [24]:
#Como antes, agregamos los actores por toma en la lista
actores_por_toma = []

for toma in tomas_actores:
    actores = set()

    for actor, aparece in enumerate(toma):
        if aparece == 1:
            actores.add(actor)

    actores_por_toma.append(actores)

#Calculamos el coste agregando los actores por toma por día.
def calcular_coste(plan):
    coste = 0

    for dia in plan:
        actores_dia = set()

        for toma in dia:
            actores_dia.update(actores_por_toma[toma])

        coste += len(actores_dia)

    return coste

#Definimos el greedy inicial
def construir_solucion_greedy():
    plan = []
    actores_por_dia = []

    # Ordenamos primero las tomas con más actores
    def numero_actores(toma):
      return len(actores_por_toma[toma])

    orden_tomas = list(range(num_tomas))
    orden_tomas.sort(key=numero_actores, reverse=True)

    for toma in orden_tomas:
        mejor_dia = None
        menor_incremento = float("inf")

        # Intentar meter la toma en un día existente
        for i in range(len(plan)):
            if len(plan[i]) < Max_tomas_dias:
                actores_nuevos = actores_por_toma[toma] - actores_por_dia[i]
                incremento = len(actores_nuevos)

                if incremento < menor_incremento:
                    menor_incremento = incremento
                    mejor_dia = i

        # Si cabe en algún día, la metemos en el mejor
        if mejor_dia is not None:
            plan[mejor_dia].append(toma)
            actores_por_dia[mejor_dia].update(actores_por_toma[toma])

        # Si no cabe, creamos un día nuevo
        else:
            plan.append([toma])
            actores_por_dia.append(actores_por_toma[toma].copy())

    return plan

#Ahora aplicaremos la busqueda local para mejorar el plan
def busqueda_local(plan):
    mejora = True

    while mejora:
        mejora = False
        coste_actual = calcular_coste(plan)

        # Probar mover una toma de un día a otro
        for origen in range(len(plan)):
            for toma in plan[origen].copy():
                for destino in range(len(plan)):

                    if origen == destino:
                        continue

                    if len(plan[destino]) >= Max_tomas_dias:
                        continue

                    plan[origen].remove(toma)
                    plan[destino].append(toma)

                    nuevo_plan = [dia for dia in plan if len(dia) > 0]
                    nuevo_coste = calcular_coste(nuevo_plan)

                    if nuevo_coste < coste_actual:
                        plan = nuevo_plan
                        mejora = True
                        break

                    plan[destino].remove(toma)
                    plan[origen].append(toma)

                if mejora:
                    break

            if mejora:
                break

        if mejora:
            continue

        # Probar intercambiar dos tomas
        for dia1 in range(len(plan)):
            for dia2 in range(dia1 + 1, len(plan)):

                for toma1 in plan[dia1].copy():
                    for toma2 in plan[dia2].copy():

                        plan[dia1].remove(toma1)
                        plan[dia2].remove(toma2)

                        plan[dia1].append(toma2)
                        plan[dia2].append(toma1)

                        nuevo_coste = calcular_coste(plan)

                        if nuevo_coste < coste_actual:
                            mejora = True
                            break

                        plan[dia1].remove(toma2)
                        plan[dia2].remove(toma1)

                        plan[dia1].append(toma1)
                        plan[dia2].append(toma2)

                    if mejora:
                        break

                if mejora:
                    break

            if mejora:
                break

    return plan

#Ejecutamos las funciones definidas
plan_inicial = construir_solucion_greedy()
coste_inicial = calcular_coste(plan_inicial)

plan_final = busqueda_local(plan_inicial)
coste_final = calcular_coste(plan_final)

print("Coste inicial greedy:", coste_inicial)
print("Coste final tras búsqueda local:", coste_final)

print("\nPlanificación final:")
for i, dia in enumerate(plan_final, start=1):
  print(f"Día {i}: tomas {[toma + 1 for toma in dia]}")

Coste inicial greedy: 37
Coste final tras búsqueda local: 28

Planificación final:
Día 1: tomas [12, 25, 5, 6, 9, 28]
Día 2: tomas [3, 10, 2, 19, 14, 4]
Día 3: tomas [21, 24, 15, 8, 11, 7]
Día 4: tomas [16, 17, 18, 20, 23, 13]
Día 5: tomas [26, 27, 29, 30, 22, 1]


In [25]:
print("Plan inicial:", plan_inicial)
print("Coste inicial:", coste_inicial)
print("Plan final:", plan_final)
print("Coste final:", coste_final)

Plan inicial: [[11, 24, 4, 5, 8, 27], [2, 9, 1, 18, 13, 3], [20, 23, 14, 7, 10, 6], [15, 16, 17, 19, 22, 12], [25, 26, 28, 29, 21, 0]]
Coste inicial: 37
Plan final: [[11, 24, 4, 5, 8, 27], [2, 9, 1, 18, 13, 3], [20, 23, 14, 7, 10, 6], [15, 16, 17, 19, 22, 12], [25, 26, 28, 29, 21, 0]]
Coste final: 28


### (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?
### ¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




**Nº posibilidades sin tener en cuenta las restricciones:**

Tenemos:

* 30 tomas.
* Dado que existen 30 tomas, el mayor número de días que podría considerarse es 30, asignando cada toma a un día distinto.

Sin imponer ninguna restricción, cada una de las 30 tomas puede asignarse de forma independiente a cualquiera de esos 30 días posibles. Por tanto, el número total de asignaciones es:

$$30^{30} \approx 2{,}06 \times 10^{44}$$

Esto cuenta también muchas soluciones equivalentes o absurdas, por ejemplo:

* días vacíos entre medias;
* planificaciones que usan más días de los necesarios;
* distribuciones que superan las 6 tomas por día;
* planificaciones idénticas salvo por cambiar el nombre de los días.

Pero como la pregunta pide calcular el número de posibilidades sin tener en cuenta las restricciones, se consideran todas las asignaciones posibles, incluso aquellas que incumplen las restricciones del problema o que representan soluciones equivalentes por un simple cambio en la numeración de los días.

En definitiva, sin considerar las restricciones, cada una de las 30 tomas puede asignarse independientemente a cualquiera de los 30 días posibles. Por tanto, el número de asignaciones posibles es $30^{30}$. Esta cifra incluye soluciones no válidas y soluciones equivalentes obtenidas al permutar el orden de los días.

**Nº posibilidades teniendo en cuenta todas las restricciones:**<br>

Al considerar las restricciones estructurales del problema, el espacio de búsqueda se reduce respecto al apartado anterior:

- Cada toma debe asignarse a un único día.
- Cada día puede contener como máximo **6 tomas**.

Dado que existen 30 tomas y cada día admite un máximo de 6, el número mínimo de días necesarios para planificar todas las grabaciones es:

$$
\frac{30}{6}=5
$$

Para realizar el cálculo, se adopta la hipótesis de que la planificación utiliza ese número mínimo de días, es decir, **5 días con 6 tomas cada uno**.

Bajo esta hipótesis, el número de formas de distribuir las 30 tomas entre los 5 días viene dado por:

$$
\frac{30!}{(6!)^5}
$$

Esta expresión se obtiene partiendo de las $30!$ formas de ordenar las 30 tomas. Sin embargo, el orden de las 6 tomas dentro de un mismo día no modifica la planificación. Como las tomas de cada día pueden ordenarse de $6!$ formas y existen 5 días, se divide entre $(6!)^5$.

Los días se consideran diferentes, es decir, día 1, día 2, día 3, día 4 y día 5, por lo que no se divide entre $5!$.

El número de planificaciones posibles bajo esta hipótesis es:

$$
\frac{30!}{(6!)^5}
=
1\,370\,874\,167\,589\,326\,400
\approx 1,37 \times 10^{18}
$$

Aunque esta cifra es muy inferior a las $30^{30} \approx 2,06 \times 10^{44}$ asignaciones posibles obtenidas sin restricciones, sigue representando un espacio de búsqueda extremadamente grande.

> **Nota:** Este cálculo corresponde únicamente a las planificaciones que utilizan el número mínimo de días. Si se permitiera emplear más días, respetando siempre el máximo de 6 tomas diarias, el número de distribuciones posibles sería mayor. Por tanto, el resultado representa el tamaño del espacio de búsqueda bajo la hipótesis de duración mínima y no el número total de planificaciones posibles del problema.

### (*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguméntalo)


Inicialmente, los datos del problema se representan mediante una **matriz binaria toma-actor**, ya que este es el formato proporcionado en el enunciado. Sin embargo, durante el desarrollo del algoritmo se observó que esta estructura no era la más adecuada para realizar los cálculos necesarios de forma eficiente.

Por este motivo, **la información se transformó en una lista de conjuntos**, donde cada toma se asocia al conjunto de actores que participan en ella. Asimismo, **la planificación se representa mediante una lista de listas**, en la que cada lista corresponde a un día de grabación y contiene las tomas asignadas.

El uso de conjuntos permite obtener rápidamente los actores presentes en un día mediante operaciones de unión y comprobar qué actores se incorporan al añadir una nueva toma, evitando recorrer continuamente la matriz binaria. Esto reduce el coste de evaluar una solución, tanto en el algoritmo de backtracking con poda como en el algoritmo heurístico basado en Greedy y búsqueda local.

Por tanto, aunque la matriz binaria resulta adecuada como formato de entrada, la combinación de lista de listas para la planificación y conjuntos para representar los actores constituye una estructura de datos más eficiente para resolver el problema.

###(*)¿Cual es la función objetivo?
###(*)¿Es un problema de maximización o minimización?

**La función objetivo es:**<br>

- **minimizar el coste total asociado a la contratación de los actores**.

Cada actor cobra por cada día en el que debe asistir al rodaje, independientemente del número de tomas que realice durante esa jornada. Por tanto, el coste total depende del número de días en los que participa cada actor.

Si denotamos por \(D(a)\) el número de días en los que interviene el actor \(a\), la función objetivo puede expresarse como:

$$
\min \sum_{a=1}^{10} D(a)
$$

donde la suma se realiza sobre los 10 actores del problema.

En el código desarrollado, el valor de la función objetivo para una planificación concreta se obtiene mediante la función `calcular_coste(plan)`. Esta función suma, para cada día de grabación, el número de actores distintos que deben asistir, obteniendo así el coste total medido en unidades **actor-día**.

Por tanto, el objetivo de los algoritmos es encontrar una planificación que minimice el valor devuelto por `calcular_coste(plan)`, respetando la restricción de un máximo de seis tomas por jornada.

**¿Maximización o minimización?**

Se trata de un **problema de minimización**, ya que el objetivo es encontrar una planificación de las tomas que reduzca al mínimo el coste total de contratación de los actores.

El coste viene determinado por el número de días en los que cada actor debe asistir al rodaje. Por tanto, una solución será mejor cuanto menor sea el número total de **actor-día**, siempre respetando la restricción de un máximo de seis tomas por jornada.

En la implementación realizada, esta minimización se refleja en la función `calcular_coste(plan)`, que devuelve el coste asociado a una planificación. Tanto el algoritmo de backtracking como el algoritmo heurístico comparan las soluciones generadas y conservan aquella cuyo coste es menor.

### Diseña un algoritmo para resolver el problema por fuerza bruta

Un algoritmo de fuerza bruta debe generar todas las planificaciones posibles de las 30 tomas, comprobar cuáles cumplen las restricciones y seleccionar aquella que tenga el menor coste.

El procedimiento sería el siguiente:

1. Generar todas las posibles asignaciones de las 30 tomas a los días disponibles.
2. Para cada asignación, construir la planificación correspondiente.
3. Comprobar que ningún día contiene más de seis tomas.
4. Descartar las planificaciones que incumplan esta restricción.
5. Calcular el coste de cada planificación válida mediante la función `calcular_coste(plan)`.
6. Comparar el coste obtenido con el mejor coste encontrado hasta ese momento.
7. Conservar la planificación cuyo coste sea mínimo.

El pseudocódigo sería:

```text
mejor_plan ← vacío
mejor_coste ← infinito

para cada asignación posible de las 30 tomas a los días:
    
    plan ← construir_planificación(asignación)
    
    si todos los días contienen como máximo 6 tomas:
        
        coste ← calcular_coste(plan)
        
        si coste < mejor_coste:
            mejor_coste ← coste
            mejor_plan ← plan

devolver mejor_plan, mejor_coste
```

Una posible implementación conceptual en Python sería:

In [ ]:
from itertools import product

def fuerza_bruta(num_tomas, num_dias, max_tomas_dia):
    mejor_plan = None
    mejor_coste = float("inf")

    # Cada posición indica el día asignado a una toma
    for asignacion in product(range(num_dias), repeat=num_tomas):

        plan = [[] for _ in range(num_dias)]

        for toma, dia in enumerate(asignacion):
            plan[dia].append(toma)

        # Comprobación de la restricción
        if any(len(dia) > max_tomas_dia for dia in plan):
            continue

        # Se eliminan los días vacíos
        plan = [dia for dia in plan if dia]

        coste = calcular_coste(plan)

        if coste < mejor_coste:
            mejor_coste = coste
            mejor_plan = plan

    return mejor_plan, mejor_coste


Este algoritmo garantiza encontrar la solución óptima porque evalúa exhaustivamente todas las asignaciones posibles. Sin embargo, su coste computacional es extremadamente elevado, ya que el número de combinaciones crece exponencialmente. Por esta razón, resulta inviable para una instancia de 30 tomas y se hace necesario emplear métodos más eficientes, como el backtracking con poda o los algoritmos heurísticos.

###Calcula la complejidad del algoritmo por fuerza bruta



El algoritmo de fuerza bruta explora todas las planificaciones posibles, evaluando cada una de ellas para determinar si cumple las restricciones y calcular su coste.

Sin considerar restricciones, cada una de las 30 tomas puede asignarse a cualquiera de los 30 días posibles. Por tanto, el número de asignaciones es:

$$
30^{30}
$$

Para cada planificación generada es necesario:

- construir la planificación;
- comprobar que ningún día supera las seis tomas;
- calcular el coste total de la solución.

Estas operaciones requieren recorrer, como máximo, las 30 tomas, por lo que su coste es lineal respecto al tamaño de la entrada, es decir, \(O(n)\).

En consecuencia, la complejidad temporal total del algoritmo viene dada por:

$$
O(30^{30}\cdot 30)
$$

Se trata de una complejidad exponencial (superexponencial respecto al crecimiento del problema), por lo que el algoritmo resulta inviable para instancias de tamaño moderado como la considerada en este trabajo. Esta es la principal motivación para emplear técnicas más eficientes, como el backtracking con poda o algoritmos heurísticos.

### (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Para mejorar el algoritmo por fuerza bruta se ha diseñado un algoritmo basado en la técnica de **Backtracking (vuelta atrás) con poda**.

A diferencia de la fuerza bruta, que explora todas las planificaciones posibles, el algoritmo de Backtracking construye la solución de forma incremental, asignando una toma en cada paso y comprobando continuamente si la solución parcial sigue siendo prometedora.

En nuestro caso, durante la construcción de la planificación se aplican varias podas:

- No se generan días con más de seis tomas.
- Se calcula el coste parcial de la planificación construida hasta ese momento.
- Si dicho coste ya es igual o superior al de la mejor solución encontrada, esa rama se descarta sin seguir explorándola.

Gracias a estas podas, muchas combinaciones dejan de analizarse antes de completarse, reduciendo considerablemente el número de soluciones evaluadas.

Aunque el peor caso sigue siendo exponencial, en la práctica el algoritmo visita una fracción muy pequeña del árbol de búsqueda respecto a la fuerza bruta, obteniendo la misma solución óptima con un tiempo de ejecución muy inferior.

Como alternativa, también se ha implementado un algoritmo Greedy, que constituye una aproximación heurística al problema. Este algoritmo reduce aún más el tiempo de ejecución al tomar decisiones locales sin explorar el resto de posibilidades, aunque, a diferencia del Backtracking con poda, no garantiza obtener la solución óptima.

### (*)Calcula la complejidad del algoritmo


El algoritmo implementado mediante **Backtracking con poda** construye la planificación de forma recursiva, asignando una toma en cada nivel del árbol de búsqueda.

En el peor de los casos, si las podas no consiguieran eliminar ninguna rama, el algoritmo tendría que explorar un número de soluciones del mismo orden que la fuerza bruta. Por ello, su complejidad temporal en el peor caso sigue siendo exponencial:

$$
O(n^n)
$$

donde \(n\) representa el número de tomas.

Sin embargo, el algoritmo incorpora diversas podas que reducen considerablemente el número de soluciones exploradas:

- no se generan planificaciones con más de seis tomas por día;
- se calcula el coste parcial de la solución;
- si dicho coste ya es igual o superior al de la mejor solución encontrada, la rama se descarta sin seguir explorándola.

Gracias a estas podas, el número de nodos visitados es mucho menor que en la fuerza bruta, por lo que el tiempo de ejecución en la práctica se reduce de forma muy significativa, aunque el peor caso continúe siendo exponencial.

La complejidad espacial viene determinada principalmente por la profundidad de la recursión y por el almacenamiento de la planificación parcial, siendo de orden:

$$
O(n)
$$

ya que, como máximo, se mantiene una asignación para cada una de las \(n\) tomas durante la exploración del árbol de búsqueda.

Por otro lado, el **algoritmo Greedy** presenta una complejidad temporal de $O(n
^2$
), ya que para cada una de las n tomas se evalúan los días disponibles para seleccionar aquel que produce el menor incremento del coste. Las operaciones realizadas sobre los conjuntos de actores tienen coste constante al existir un número fijo de actores. La complejidad espacial es $O(n)$, debido al almacenamiento de la planificación y de los conjuntos de actores asociados a cada jornada.

### Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

En este problema no resulta especialmente útil generar un juego de datos aleatorio, ya que el propio enunciado proporciona una instancia concreta del problema mediante una matriz binaria que indica qué actores participan en cada una de las 30 tomas.

Esta matriz constituye la entrada sobre la que deben trabajar los algoritmos desarrollados y permite comparar los resultados obtenidos por las distintas estrategias de resolución (Backtracking y algoritmo heurístico) utilizando exactamente las mismas condiciones.

Aunque sería posible generar de forma aleatoria nuevas matrices de participación entre actores y tomas para realizar pruebas adicionales de rendimiento o escalabilidad, este tipo de datos no forma parte del problema planteado y podría producir instancias con características muy diferentes a las previstas en el enunciado. Por ello, en este trabajo se ha utilizado exclusivamente la matriz de datos proporcionada.

### Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

- Universidad Internacional de Valencia (VIU). *Material docente de la asignatura 03MIAR – Algoritmos de Optimización*.

- Universidad Internacional de Valencia (VIU). *Material docente de la asignatura 01MIAR – Python para la Inteligencia Artificial*.

- Python Software Foundation. (2026). *Python 3 Documentation*. https://docs.python.org/3/

- Python Software Foundation. (2026). *PEP 8 – Style Guide for Python Code*. https://peps.python.org/pep-0008/


###Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Como posibles líneas de trabajo futuras se plantean:

- Evaluar el comportamiento de los algoritmos sobre instancias de mayor tamaño.
- Diseñar nuevas heurísticas que proporcionen soluciones de alta calidad con menor tiempo de ejecución.
- Incorporar técnicas de metaheurística, como algoritmos genéticos, recocido simulado o búsqueda tabú.
- Estudiar estrategias de poda más eficientes que reduzcan aún más el espacio de búsqueda del algoritmo de Backtracking.
- Analizar el compromiso entre calidad de la solución y tiempo de ejecución en problemas de mayor complejidad.